### SAMPLE LANG GRAPH

In [ ]:
# simple graph
# play around

import os
import random # for radom method
from dotenv import load_dotenv # load evn
from langgraph.graph import StateGraph, START, END 
from typing_extensions import TypedDict 
from typing import Literal # constant
from IPython.display import Image,display # display the graph
load_dotenv()  # loads environment variables from .env


class State(TypedDict):
    msg: str

graph_builder = StateGraph(State)

# define nodes
def start_play(state: State):
    #print("Inside the start payling")
    #print(state["msg"])
    #msg_list = state["msg"] + ["i am planning to play."]
    #return {"msg": msg_list }
    return {"msg": state["msg"]+ "i am planning to play." }

def circket_play(state: State):
    #print("Inside the circket payling")
    msg_list = state["msg"] + ["i am playing cricket."]
    #return {"msg": msg_list}
    return {"msg": state["msg"]+ "i am playing cricket." }

def batminton_play(state: State):
    #print("Inside the badminton payling")
    return {"msg": state["msg"]+ "i am playing badminton." }
    #msg_list = state["msg"] + ["i am playing badminton."]
    #return {"msg": msg_list }

def random_pick(state: State)-> Literal["cricket", "badminton", END]:
    value = random.random()
    #print(f"random value is {value}")
    if value > 0.5:
        return "cricket"
    elif value > 0.2:
        return "badminton"
    else:
        return END


# build graph and nodes
graph_builder.add_node("start_play", start_play)
graph_builder.add_node("cricket", circket_play)
graph_builder.add_node("badminton", batminton_play)

# add connection/flow of the graph
graph_builder.add_edge(START, "start_play")
graph_builder.add_conditional_edges("start_play", random_pick)

graph = graph_builder.compile()

#View the graph
display(Image(graph.get_graph().draw_mermaid_png()))

#inoke graph with default state
#for chunk in graph.stream({"msg": "I am suresh. "}, stream_mode="updates"):
    #print(chunk)


async for chunk in graph.astream_events({"msg": "I am suresh. "}, stream_mode="updates"):
    print(chunk)



### Chatbot using langraph

In [ ]:
from typing import Annotated ## for labelling the messages
from langgraph.graph.message import add_messages ## reducers for messages
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

## langgraph state/model creation
class State(TypedDict):
    messages: Annotated[list, add_messages]

load_dotenv()  # loads environment variables from .env
llm = ChatGroq(model_name="openai/gpt-oss-20b", api_key=os.getenv("GROQ_API_KEY"))

def chat_node(state:State) -> State:
    print("I am in chat node")
    return {"messages": llm.invoke(state["messages"])}

graph = StateGraph(State)

graph.add_node("chat", chat_node)

graph.add_edge(START, "chat")
final_graph = graph.compile()

#View the graph
display(Image(final_graph.get_graph().draw_mermaid_png()))

print(final_graph.invoke({"messages": "hi, who are you?"}))



In [ ]:
from pydantic import BaseModel
## Advanced compare to typedict, throws runtime error

class User(BaseModel):
    name: str
    age: int

u = User(name='Alice', age=12)
print(u)

### Langchain diff type of messages

In [ ]:
### LANGCHAIN MESSAGES

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from pprint import pprint

msgs = [SystemMessage(content="you are the AI Agent, to help the user queires", name="system")]
msgs.append(AIMessage(content = "How can i help you today", name="AI"))
msgs.append(HumanMessage(content = "help me to learn the famous full stack", name="user"))

for msg in msgs:
    print(msg)
    msg.pretty_print()


load_dotenv()  # loads environment variables from .env
llm = ChatGroq(model_name="openai/gpt-oss-20b", api_key=os.getenv("GROQ_API_KEY"))

print(llm.invoke(msgs))


In [ ]:
from typing import Annotated ## for labelling the messages
from langchain_core.tools import tool
from langgraph.graph.message import add_messages ## reducers for messages
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, AnyMessage
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# loads environment variables from .env
load_dotenv()  
llm = ChatGroq(model_name="openai/gpt-oss-20b", api_key=os.getenv("GROQ_API_KEY"))

# custom add_message
def my_add_messages(prev, new):
    #print(f"my_add_msg prev: {prev} new: {new}")
    if prev is None:
        prev = []
    if new is None:
        return prev

    # if new is a single item, convert to list
    if not isinstance(new, list):
        new = [new]

    return prev + new

class State(TypedDict):
    messages: Annotated[list, my_add_messages]


# define tools
@tool(parse_docstring=True)
def add(a: int, b: int) -> int:
    """
    Add two integers.

    Args:
        a: The first number.
        b: The second number.

    Returns:
        The sum of a and b.
    """
    return a + b

@tool(parse_docstring=True)
def multiply(a: int, b: int) -> int:
    """
    multiply two integers.

    Args:
        a: The first number.
        b: The second number.

    Returns:
        The multiply of a and b.
    """
    return a * b


tools = [add, multiply]
tool_node = ToolNode(tools=tools)
llm_with_tool = llm.bind_tools(tools)

# define node
def chat_node(state:State) -> State:
    return {"messages": llm_with_tool.invoke(state["messages"])}
    

graph_builder = StateGraph(State)

graph_builder.add_node("chat", chat_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chat")
graph_builder.add_conditional_edges("chat", tools_condition)
graph_builder.add_edge("tools", "chat")
graph = graph_builder.compile()

#View the graph
display(Image(graph.get_graph().draw_mermaid_png()))

system_role = SystemMessage(content=f"you are helpful math assistant, you can do calucation using tools, don't answer any other questions appart from maths ")
response = graph.invoke({"messages": [
    system_role,
    "1%2?"
]})

for msg in response["messages"]:
    #msg.pritty_print()
    print(msg)

### CLI Chatbot
 Langgraph use a check pointer to automatically sae the graph state after each step. 

In [ ]:
#cli chatbot

from langgraph.checkpoint.memory import MemorySaver

# to make the memory across the sessions need (memory = MemorySaver())
memory = MemorySaver()
graph = graph_builder.compile(memory)


thread_id = "session_002"
while True:
    input_msg = input("You: ")
    if input_msg in {"quit", "exit"}:
        break
    response = graph.invoke({"messages": input_msg}, config={"thread_id": thread_id})
    # last message from the history
    chat_history = response["messages"][-1]
    chat_history.pretty_print()

### Streaming


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# to make the memory across the sessions need (memory = MemorySaver())
memory = MemorySaver()
graph = graph_builder.compile(memory)


thread_id = "session_003"
while True:
    input_msg = input("You: ")
    if input_msg in {"quit", "exit"}:
        break
    # we have "updates" and "values"
    response = graph.stream({"messages": input_msg}, config={"thread_id": thread_id}, stream_update="updates")
    for chunk in response:
        print(chunk)


### Debugging and deploying
